# 1. Synthetic-stage purpose and research questions

The synthetic stage asks whether an externally imposed event effect can be detected; whether predictive direction and the first structural lag can be distinguished; whether the event-response multiplier can be estimated; whether event-aware models improve volatility prediction; and how these results change with parameters, functionals, powered states and schedules.

The frozen convention is

\[
E_i \longrightarrow \omega_{i+1} \longrightarrow X_{i+1}.
\]

Event age 0 affects the first next-state observation; ages 0-12 correspond to direct predictive lags +1 through +13. Known-DGP quantities are retrospective synthetic references only.

# 2. Synthetic experiment map

The map distinguishes deterministic validation, one-seed illustration, repeated-seed robustness, and paired repeated-seed comparison. Notebook 06 supplies reusable infrastructure rather than a standalone scientific result.

In [1]:
from pathlib import Path
import json, re
import nbformat
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
paths = {
    '08': ROOT / 'notebook08_results' / 'notebook08_numeric_summary.csv',
    '09': ROOT / 'notebook09_results' / 'notebook09_calibration_numeric_summary.csv',
    '10runs': ROOT / 'notebook10_results' / 'stage4_seed_case_analysis.csv',
    '10notebook': ROOT / '10_event_frequency_and_schedule_robustness.ipynb',
    '11': ROOT / 'notebook11_results' / 'run_metrics.csv',
    '12': ROOT / '12_log_state_event_decomposition.ipynb',
}
assert all(path.exists() for path in paths.values())
summary08 = pd.read_csv(paths['08'])
summary09 = pd.read_csv(paths['09'])
schedule_runs = pd.read_csv(paths['10runs'])
run_metrics = pd.read_csv(paths['11'])
nb10 = nbformat.read(paths['10notebook'], as_version=4)
nb12 = nbformat.read(paths['12'], as_version=4)
assert not any(output.get('output_type') == 'error' for cell in nb12.cells for output in cell.get('outputs', []))
nb12_text = '\n'.join(
    ''.join(output.get('data', {}).get('text/markdown', []))
    for cell in nb12.cells for output in cell.get('outputs', [])
    if output.get('output_type') in {'display_data', 'execute_result'}
)
assert 'mostly inferior' in nb12_text
nb10_text = '\n'.join(
    ''.join(output.get('data', {}).get('text/markdown', []))
    for cell in nb10.cells for output in cell.get('outputs', [])
    if output.get('output_type') in {'display_data', 'execute_result'}
)
assert all(label in nb10_text for label in ['Dense periodic finding:', 'Sparse periodic finding:', 'Sparse irregular finding:'])
experiment_map = pd.DataFrame([
    ('00', 'Simulator, timing and information structure', 'Deterministic validation', 'final'),
    ('01-05', 'Benchmark detection, recovery and forecasting', 'Canonical one-path workflow', 'final'),
    ('06', 'Reusable workflow infrastructure', 'No standalone scientific result', 'final'),
    ('07', 'Nonlinear functional-form stress test', 'Exploratory one-seed evidence', 'final exploratory'),
    ('08', 'Scalar-parameter and replication robustness', '50-seed repeated evidence', 'final'),
    ('09', 'n=2 and scale-matched comparisons', 'Mixed illustrative and repeated evidence', 'final'),
    ('10', 'Event-frequency and schedule robustness', '50 paired seeds', 'final'),
    ('11', 'Dependence and causality comparison', '50-seed repeated comparison', 'final'),
    ('12', 'Log-state representation', 'Matched one-seed comparison', 'final exploratory'),
], columns=['notebook', 'main question', 'evidence type', 'status'])
display(experiment_map)

,notebook,main question,evidence type,status
0,00,"Simulator, timing and information structure",Deterministic validation,final
1,01-05,"Benchmark detection, recovery and forecasting",Canonical one-path workflow,final
2,06,Reusable workflow infrastructure,No standalone scientific result,final
3,07,Nonlinear functional-form stress test,Exploratory one-seed evidence,final exploratory
4,08,Scalar-parameter and replication robustness,50-seed repeated evidence,final
5,09,n=2 and scale-matched comparisons,Mixed illustrative and repeated evidence,final
6,10,Event-frequency and schedule robustness,50 paired seeds,final
7,11,Dependence and causality comparison,50-seed repeated comparison,final
8,12,Log-state representation,Matched one-seed comparison,final exploratory


# 3. Detection, direction and lag timing

GC provides linear directional predictive evidence when the distributed-lag model and history window are appropriate. A cumulative GC order \(p\) jointly contains lags \(1,\ldots,p\), so a selected order is not an isolated causal lag. Individual conditional-lag tests instead identify the strongest distinguishable direct lag.

TDMI describes unsigned nonlinear lagged dependence. Correlation and TDMI can look bidirectional because they do not condition on target history in the same way as GC or TE. Transfer entropy is directional in principle but data-hungry and finite-sample sensitive. Detection is easier than structural-lag recovery; dense periodic calendars also create wider-lag aliases and reverse predictive artefacts. None of GC, TDMI, or TE proves interventionist structural causality.

In [2]:
expected_cases = {'Dense periodic (n=1)', 'Sparse irregular (n=1)', 'State-dependent nu (n=1)', 'Locally scale-matched (n=2)', 'No effect (n=1)'}
methods = ['Lagged correlation', 'Granger causality p=1', 'Granger causality p=13', 'TDMI', 'Transfer entropy']
assert set(run_metrics.case_name) == expected_cases and set(run_metrics.method) == set(methods)
assert run_metrics.groupby(['case_name','method','transformation']).size().eq(50).all()
signal = run_metrics.loc[(run_metrics.transformation == 'changes') & (run_metrics.case_name != 'No effect (n=1)')]
method_evidence = []
for method in methods:
    d = signal.loc[signal.method.eq(method)]
    exact = np.nan if method == 'Granger causality p=13' else d.exact_plus_1_recovery.mean()
    method_evidence.append((
        method.replace('Granger causality ', 'GC ').replace('Transfer entropy', 'TE'),
        d.forward_detected.mean(), d.directional_classification.eq('forward only').mean(), exact,
        d.direct_window_recovery.mean(),
        'joint direct-window test; unique lag not identified' if method == 'Granger causality p=13' else ('prespecified +1 GC evidence' if method == 'Granger causality p=1' else 'peak-based structural-lag recovery'),
    ))
method_evidence = pd.DataFrame(method_evidence, columns=['method','forward detection','forward-only classification','exact +1 recovery','direct-window recovery','timing interpretation'])
display(method_evidence)
assert method_evidence.loc[method_evidence.method.eq('GC p=13'), 'exact +1 recovery'].isna().all()

,method,forward detection,forward-only classification,exact +1 recovery,direct-window recovery,timing interpretation
0,Lagged correlation,0.950,0.195,0.520,0.540,peak-based structural-lag recovery
1,GC p=1,0.975,0.575,0.975,0.975,prespecified +1 GC evidence
2,GC p=13,1.000,0.420,NaN,1.000,joint direct-window test; unique lag not ident...
3,TDMI,0.715,0.140,0.430,0.410,peak-based structural-lag recovery
4,TE,0.410,0.045,0.540,0.410,peak-based structural-lag recovery


# 4. Event-profile recovery

The benchmark profile is deliberately smooth and exponential. Under that studied design, the parametric estimator generally benefits from the correctly specified shape, whereas the non-parametric estimator is more flexible but noisier. Sparse schedules reduce observations per event age and weaken recovery. Both estimates depend on the event-free background transition, and good forecasting does not by itself establish accurate structural-profile recovery.

No final executed Notebook 08 result supports an event-specific-weight claim, so none is made here.

# 5. Forecasting and calibration

Event-aware forecasts improve active-event-window performance in the synthetic benchmark. Event amplitude and volatility-innovation scale change signal-to-noise and forecast gains; sparse events generally reduce profile precision and improvement. Representative paths are pedagogical, whereas repeated-seed distributions are robustness evidence.

Forecasting is technically closer to a delayed-volatility nowcast: the next spot observation is available before the next volatility state is treated as observed. Calibration requires empirical coverage, coverage error, interval width and interval score together; 100% coverage alone is not necessarily good calibration.

# 6. Robustness findings

Notebook 07 is a focused one-seed stress test, not general nonlinear robustness. Notebook 08 supplies repeated 50-seed parameter evidence: amplitude drives forecast gains, innovation scale governs signal-to-noise, and decay controls persistence. In Notebook 09, \(X_i=\sigma_i^2\) for \(n=2\) and the multiplier acts directly on the powered state; dedicated calibrations are not a pure test of \(n\), while scale-matched comparisons remain calibration-specific.

Notebook 10's final paired evidence is used below. Notebook 11 separates detection, directional classification and structural-lag recovery: GC p=1 matches the prespecified lag, GC p=13 is joint, TDMI/correlation detect dependence but struggle with direction, and TE is theoretically directional but weak in finite sparse samples. Notebook 12's conclusion is specific to its matched-feature linear implementation.

In [3]:
def metric_mean(frame, case, metric):
    value = frame.loc[(frame.case_name == case) & (frame.metric == metric), 'mean']
    assert len(value) == 1
    return float(value.iloc[0])

schedule = schedule_runs.groupby('case_name').agg(
    profile_rmse=('parametric_profile_RMSE','mean'),
    active_mae_gain=('active_sigma_parametric_MAE_gain_pct','median'),
    coverage_error=('parametric_absolute_coverage_error','mean'),
).reset_index()
# Final Notebook 10 execution supplies the corrected forward-GC and modal-direct-lag values.
findings = {}
for label, key in [('Dense periodic','dense_periodic'),('Sparse periodic','sparse_periodic'),('Sparse irregular','sparse_irregular')]:
    match = re.search(rf'{label} finding: forward linear directional predictive evidence rejects in (\d+)%.*?modal strongest direct individual lag is (\d+)', nb10_text)
    assert match, f'Missing final Notebook 10 finding for {label}'
    findings[key] = (int(match.group(1)), int(match.group(2)))
robustness = pd.DataFrame([
    ('07 nonlinear functionals', 'Frozen workflow remains usable in a focused stress test; numerical result omitted.', 'one-seed exploratory', 'not general nonlinear robustness'),
    ('08 scalar parameters', f"Repeated n=1 benchmark: active MAE gain {metric_mean(summary08,'benchmark','parametric_active_MAE_gain_pct'):.1f}%.", '50 seeds per case', 'chosen parameter grid'),
    ('09 powered states', f"Scale-matched n=2 active MAE gain {metric_mean(summary09,'n2_benchmark','parametric_active_MAE_gain_pct'):.1f}%.", '50 repeated seeds', 'not a universal n=2 result'),
    ('10 schedules', f"Forward GC: dense/sparse/irregular {findings['dense_periodic'][0]}%/{findings['sparse_periodic'][0]}%/{findings['sparse_irregular'][0]}%; modal direct lag 1 in each.", '50 paired seeds', 'three studied calendars'),
    ('11 methods', 'Detection, direction and structural-lag recovery are different tasks; no universal ranking.', '50 repeated seeds', 'method and representation dependence'),
    ('12 log state', 'Mostly inferior under the matched-feature raw linear predictor basis.', 'one-seed exploratory', 'not all log-scale models'),
], columns=['study','main conclusion','evidence strength','key limitation'])
display(robustness)
assert schedule.case_name.tolist() == ['dense_periodic','sparse_irregular','sparse_periodic']
assert all(value[1] == 1 for value in findings.values())

,study,main conclusion,evidence strength,key limitation
0,07 nonlinear functionals,Frozen workflow remains usable in a focused st...,one-seed exploratory,not general nonlinear robustness
1,08 scalar parameters,Repeated n=1 benchmark: active MAE gain 22.8%.,50 seeds per case,chosen parameter grid
2,09 powered states,Scale-matched n=2 active MAE gain 71.1%.,50 repeated seeds,not a universal n=2 result
3,10 schedules,Forward GC: dense/sparse/irregular 98%/94%/90%...,50 paired seeds,three studied calendars
4,11 methods,"Detection, direction and structural-lag recove...",50 repeated seeds,method and representation dependence
5,12 log state,Mostly inferior under the matched-feature raw ...,one-seed exploratory,not all log-scale models


# 7. What the synthetic evidence establishes

1. Event-related predictive dependence can be detected under the studied synthetic design.
2. The first structural effect occurs at lag +1.
3. Distributed persistence makes structural-lag recovery harder than detection.
4. A smooth parametric profile is advantageous when the true profile is exponential.
5. Event-aware forecasts can improve active-window prediction.
6. Amplitude, noise scale, event spacing and sample size materially affect performance.
7. Periodic schedules can generate aliases.
8. Repeated-seed evidence is needed to distinguish structural findings from path-specific noise.

# 8. What the synthetic evidence does not establish

It does not establish interventionist causality from GC, TDMI or TE alone; universally correct lag recovery; universal superiority of any dependence measure, \(n=2\), original-state models, or log-state models; or that good forecasts recover the true event mechanism.

Latent synthetic volatility is not observable ATM implied volatility. Empirically, neither the true event-free counterfactual transition nor the true multiplier is observed. One-seed results remain exploratory; repeated results remain conditional on the calibration. The benchmark profile and event timing are deliberately cleaner than real announcement and quote timing.

# 9. Synthetic-to-empirical transition

The synthetic estimand is

\[
X_{t+1}=\omega_{t+1}^{1/n}F_t,\qquad \omega(a)>0.
\]

The empirical notebooks instead use signed additive event coefficients in ATM implied-volatility changes. Latent physical volatility is not observed, ATM IV is not identical to the simulator state, hidden shocks and the true background transition are unavailable, the multiplier is unobserved, and empirical event/quote timing can be uncertain.

The intended research chain is

\[
\text{structural synthetic model} \rightarrow \text{observable-data estimator validated synthetically} \rightarrow \text{real FX-options application}.
\]

A candidate bridge, not a frozen final specification, is

\[
\Delta\log X_t=\text{background terms}+\sum_a\gamma_aD_{t,a}+u_t,\qquad \widehat\omega(a)=\exp(\widehat\gamma_a).
\]

Ordinary \(\Delta IV_t\) remains an interpretable robustness specification. **Notebook 14 therefore begins with a real-data audit rather than immediately transferring the synthetic estimator.**